<pre> <font color="#e087c8">   ____   _ </font>
<font color="#e087c8">   / __ \ (_)  ____   _____ </font>
<font color="#e087c8">  / /_/ / / / / __ \ / ___/ </font>       Probability of interface native contacts
<font color="#e087c8"> / ____/ / / / / / // /__  </font>    / an interpretable AlphaFold interaction score /
<font color="#e087c8">/_/     /_/ /_/ /_/ \___/ </font> </pre>

In [ ]:
#@markdown ### <b><font>Upload files</font></b>
#@markdown ##### <font color='#2656c9'>You will be prompted to upload the structure file (PDB/mmCIF) and its PAE matrix (JSON)</font>

import os
import sys
import zipfile
import pandas as pd
import rpy2
import rpy2.robjects as robjects
from google.colab import files, data_table
from io import StringIO
from IPython.display import clear_output, SVG, display

if "rpy2.ipython" not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic("load_ext", "rpy2.ipython")

data_table.enable_dataframe_formatter()

print("Upload the structure file (PDB/mmCIF):")
up1 = files.upload()
if len(up1) != 1:
    raise ValueError("Please upload exactly one structure file.")

str_file = next(iter(up1))
str_ext = os.path.splitext(str_file)[1].lower()
if str_ext not in [".pdb", ".cif", ".mmcif"]:
    raise ValueError(f"Unsupported structure file extension: {str_ext} (expected .pdb, .cif, or .mmcif)")

print("Upload the PAE matrix (JSON):")
up2 = files.upload()
if len(up2) != 1:
    raise ValueError("Please upload exactly one JSON file.")

jsn_file = next(iter(up2))
jsn_ext = os.path.splitext(jsn_file)[1].lower()
if jsn_ext != ".json":
    raise ValueError(f"Unsupported JSON file extension: {jsn_ext} (expected .json)")

%Rpush jsn_file str_file
%R if (!requireNamespace('jsonlite', quietly=TRUE)) install.packages('jsonlite', repos='https://cloud.r-project.org')

r_script = r"""
CONTACT_RADIUS <- 12

parse_pae <- function(jsn_file) {
  jsn <- jsonlite::fromJSON(jsn_file)
  if (!is.null(jsn$pae)) {
    return(jsn$pae)
  }
  if (!is.null(jsn$predicted_aligned_error)) {
    return(jsn$predicted_aligned_error)
  }
  stop(
    "No PAE matrix found in JSON. Expected field 'pae' or 'predicted_aligned_error'."
  )
}

reduce_to_com <- function(coords) {
  xyz <- c('x', 'y', 'z')
  coords$m <- c('S' = 32.0650, 'P' = 30.9738, 'O' = 15.9994, 'N' = 14.0067)[
    coords$e
  ]
  coords$m[is.na(coords$m)] <- 12.0107

  grp <- paste0(coords$c, '_', coords$r)
  grp <- factor(grp, levels = unique(grp))

  com <- aggregate(coords[, xyz] * coords$m, by = list(grp), FUN = sum)
  com[, xyz] <- com[, xyz] / tapply(coords$m, grp, sum)
  com
}

str_to_dist <- function(str_file) {
  lines <- readLines(str_file, warn = FALSE)
  atom_idx <- grep('^ATOM|^HETATM', lines)

  if (length(atom_idx) < 10) {
    stop('Non-structure file or insufficient number of ATOM records.')
  }

  if (any(grepl('_atom_site.group_PDB', lines))) {
    start <- grep('_atom_site.group_PDB', lines)
    records <- trimws(lines[start:(atom_idx[1] - 1)])
    header <- c(
      c = '_atom_site.label_asym_id',
      r = '_atom_site.label_seq_id',
      e = '_atom_site.type_symbol',
      x = '_atom_site.Cartn_x',
      y = '_atom_site.Cartn_y',
      z = '_atom_site.Cartn_z'
    )

    keep <- records %in% header
    coords <- read.table(textConnection(lines[atom_idx]), na.strings = NULL)[
      keep
    ]
    colnames(coords) <- names(header)[match(records[keep], header)]

    coords$r <- ave(coords$r, coords$c, FUN = function(x) {
      m <- x == '.'
      if (any(m)) {
        x[m] <- 1:sum(m)
      }
      x
    })
  } else {
    atom <- lines[atom_idx]
    coords <- data.frame(
      c = trimws(substr(atom, 22, 22)),
      r = trimws(substr(atom, 23, 26)),
      e = trimws(substr(atom, 77, 78)),
      x = as.numeric(substr(atom, 31, 38)),
      y = as.numeric(substr(atom, 39, 46)),
      z = as.numeric(substr(atom, 47, 54))
    )
  }

  com <- reduce_to_com(coords[coords$e != 'H', ])
  dist_mat <- as.matrix(dist(com[c('x', 'y', 'z')], upper = TRUE))

  colnames(dist_mat) <- sub('_.*$', '', com[, 1])
  rownames(dist_mat) <- com[, 1]
  dist_mat
}

pae_to_prob <- function(pae_mat, dist_mat) {
  vol_sphere <- function(r) (4 / 3) * pi * (r^3)

  vol_intersect <- function(Ru, D) {
    vol_vec <- numeric(length(Ru))
    ok <- is.finite(Ru) & is.finite(D) & (Ru > 0)

    if (!any(ok)) {
      return(vol_vec)
    }

    rc <- CONTACT_RADIUS
    ru <- Ru[ok]
    dd <- D[ok]
    vol <- numeric(length(dd))

    disjoint <- dd >= (rc + ru)

    contain <- dd <= abs(rc - ru)
    if (any(contain)) {
      contain[is.na(contain)] <- FALSE
      vol[contain] <- vol_sphere(pmin(rc, ru[contain]))
    }

    other <- !(contain | disjoint)
    if (any(other)) {
      other[is.na(other)] <- FALSE
      ddo <- dd[other]
      ruo <- ru[other]
      vol[other] <- pmax(
        0,
        pi *
          (rc + ruo - ddo)^2 *
          (ddo^2 + 2 * ddo * (ruo + rc) - 3 * (ruo - rc)^2) /
          (12 * ddo)
      )
    }

    vol_vec[ok] <- vol
    vol_vec
  }

  n <- nrow(pae_mat)
  unc_ij <- as.numeric(t(pae_mat))

  dist_vec <- as.numeric(t(dist_mat))
  vol_unc_ij <- vol_sphere(unc_ij)

  p_ij <- vol_intersect(unc_ij, dist_vec) / vol_unc_ij

  p_ij[!is.finite(p_ij)] <- 0
  p_ij <- pmax(0, pmin(1, p_ij))

  cont_asym <- t(matrix(p_ij, nrow = n, ncol = n))
  diag(cont_asym) <- 1
  colnames(cont_asym) <- colnames(dist_mat)
  rownames(cont_asym) <- rownames(dist_mat)

  cont_asym
}

compute_pinc <- function(cont_asym, dist_mat) {
  chains <- colnames(dist_mat)
  uchain <- unique(chains)
  idx <- lapply(uchain, function(x) which(chains == x))
  names(idx) <- uchain

  pairs <- t(combn(uchain, 2))
  pinc1 <- numeric(nrow(pairs))
  pinc2 <- numeric(nrow(pairs))

  for (k in seq_len(nrow(pairs))) {
    i <- idx[[pairs[k, 1]]]
    j <- idx[[pairs[k, 2]]]

    dvec <- dist_mat[i, j, drop = FALSE]
    keep <- is.finite(dvec) & (dvec < CONTACT_RADIUS)

    if (any(keep)) {
      pinc1[k] <- mean(cont_asym[j, i][t(keep)])
      pinc2[k] <- mean(cont_asym[i, j][keep])
    }
  }

  pinc_df <- data.frame(
    chain1 = pairs[, 1],
    chain2 = pairs[, 2],
    Pinc1 = sprintf('%.4f', pinc1),
    Pinc2 = sprintf('%.4f', pinc2),
    Pinc = sprintf('%.4f', (pinc1 + pinc2) / 2)
  )

  pinc_df <- pinc_df[order(pinc_df$Pinc, decreasing = TRUE), ]
  rownames(pinc_df) <- NULL
  pinc_df
}

contact_pairlist <- function(cont_asym, dist_mat) {
  cont_sym <- (cont_asym + t(cont_asym)) / 2

  keep <- upper.tri(cont_sym, diag = FALSE) &
    (cont_sym > 0) &
    (dist_mat < CONTACT_RADIUS) &
    (colnames(cont_sym)[row(cont_sym)] != colnames(cont_sym)[col(cont_sym)])

  ij <- which(keep, arr.ind = TRUE)
  pair_df <- data.frame(
    token1 = rownames(cont_sym)[ij[, 1]],
    token2 = rownames(cont_sym)[ij[, 2]],
    contact_p = sprintf('%.4f', cont_sym[keep]),
    distance = sprintf('%.4f', dist_mat[keep])
  )

  pair_df[order(pair_df$contact_p, decreasing = TRUE), ]
}

draw_heatmap <- function(x, base_name) {
  stopifnot(is.matrix(x))

  n <- nrow(x)
  cols <- colorRampPalette(c('#ffffff', '#024e1f'))(100)

  chain <- colnames(x)
  if (is.null(chain)) {
    chain <- rep('', n)
  }

  starts <- c(1, which(chain[-1] != chain[-length(chain)]) + 1, n)
  starts <- sort(unique(starts))

  tick_labels <- ifelse(
    starts == n,
    as.character(n),
    paste0(chain[starts], ':1')
  )
  tick_labels[length(tick_labels)] <- ''

  svg(filename = paste0(base_name, '.svg'), width = 7, height = 7)

  layout(matrix(c(1, 2), nrow = 2), heights = c(4, 0.7))
  par(mar = c(4, 4, 2, 2), mgp = c(2.5, 0.7, 0))

  image(
    t(x[n:1, ]),
    useRaster = TRUE,
    col = cols,
    axes = FALSE,
    xlab = 'Scored residue',
    ylab = 'Aligned residue',
    zlim = c(0, 1)
  )

  axis(1, at = (starts - 1) / (n - 1), labels = tick_labels, cex.axis = 0.7)
  axis(
    2,
    at = (n - starts) / (n - 1),
    labels = tick_labels,
    las = 2,
    cex.axis = 0.7
  )

  par(mar = c(2.5, 4, 0.2, 2), mgp = c(1.5, 0.5, 0))

  plot.new()
  plot.window(xlim = c(0, 1), ylim = c(0, 1))
  rasterImage(
    as.raster(matrix(cols, nrow = 1)),
    xleft = 0,
    ybottom = 0.2,
    xright = 1,
    ytop = 0.8
  )
  rect(0, 0.2, 1, 0.8, lwd = 1)
  axis(1, at = c(0, 0.5, 1), labels = c(0, 0.5, 1), tck = -0.15, cex.axis = 0.8)
  mtext('Contact probability', side = 1, line = 1.2)
  layout(1)

  dev.off()
}

pae_mat <- parse_pae(jsn_file)
dist_mat <- str_to_dist(str_file)

if (!identical(dim(dist_mat), dim(pae_mat))) {
  stop(
    'Dimension mismatch between distance and PAE matrices.\n',
    'Distance matrix dimensions: ',
    paste(dim(dist_mat), collapse = ' x '),
    '\n',
    'PAE matrix dimensions: ',
    paste(dim(pae_mat), collapse = ' x '),
    '\n',
    'Check that the structure file and PAE matrix correspond to the same model.\n'
  )
}

cont_asym <- pae_to_prob(pae_mat, dist_mat)

base_name <- tools::file_path_sans_ext(tolower(basename(str_file)))

pinc_df <- compute_pinc(cont_asym, dist_mat)

write.csv(
  x = pinc_df,
  file = paste0(base_name, '_Pinc.csv'),
  row.names = FALSE,
  quote = FALSE
)

jsonlite::write_json(
  x = list(
    token_chain_ids = colnames(cont_asym),
    contact_probability = cont_asym
  ),
  path = paste0(base_name, '_contact_probability.json'),
  digits = 4,
  pretty = TRUE
)

write.csv(
  x = contact_pairlist(cont_asym, dist_mat),
  file = paste0(base_name, '_pairlist.csv'),
  row.names = FALSE,
  quote = FALSE
)

draw_heatmap((cont_asym + t(cont_asym)) / 2, base_name)
"""

robjects.r(r_script)
clear_output()

In [ ]:
#@markdown ### <b><font>Display the contact probability matrix</font></b>
#@markdown ##### <font color='#2656c9'>A PAE plot-like heatmap of contact probabilities</font>

try:
    base_name = os.path.splitext(os.path.basename(str_file.lower()))[0]
    svg_path = f"{base_name}.svg"

    if not os.path.exists(svg_path):
        raise FileNotFoundError(f"Missing file: {svg_path}")

    display(SVG(svg_path))

except Exception:
    print("An error occurred. Try running the first code chunk again.")

In [ ]:
#@markdown ### <b><font>Display Pinc scores for all chain pairs</font></b>
#@markdown ##### <b><font color='#e087c8'>Pinc</font></b><font color='#2656c9'>: Probability of interface native contacts</font>
%R pinc_df

In [ ]:
#@markdown ### <b><font>Download a zipped folder with the results</font></b>
#@markdown ##### <font color='#2656c9'>- Pinc scores for all chain pairs (CSV)</font>
#@markdown ##### <font color='#2656c9'>- Token-level non-zero contact probabilities between interacting chains (CSV)</font>
#@markdown ##### <font color='#2656c9'>- Contact probability matrix (JSON, SVG)</font>

base_name = os.path.splitext(os.path.basename(str_file.lower()))[0]

result_files = [
    f"{base_name}_pairlist.csv",
    f"{base_name}_contact_probability.json",
    f"{base_name}_Pinc.csv",
    f"{base_name}.svg"
]

missing = [p for p in result_files if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Some expected result files were not found:\n"
        + "\n".join(missing)
        + "\n\nMake sure you ran the Pinc calculation chunk successfully."
    )

zip_name = f"{base_name}_results.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in result_files:
        zf.write(p, arcname=os.path.basename(p))

files.download(zip_name)